In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Amazon Musical Instruments EDA") \
    .getOrCreate()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1783332931503_0001,pyspark3,idle,Link,Link,✔


SparkSession available as 'spark'.


In [2]:
spark

In [18]:
df_review = spark.read.json(
    "s3://amazon-raw-data-group2/Musical_Instruments.jsonl"
)
df_review.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

In [19]:
print(df_review.count())

3017439

In [17]:
import pyspark.sql.functions as F
df_meta = spark.read.text("s3://amazon-raw-data-group2/meta_Musical_Instruments.jsonl")
df_meta = df_meta.select(
    F.get_json_object(F.col("value"), "$.main_category").alias("main_category"),
    F.get_json_object(F.col("value"), "$.title").alias("product_name"),
    F.get_json_object(F.col("value"), "$.average_rating").alias("average_rating"),
    F.get_json_object(F.col("value"), "$.rating_number").alias("rating_number"),
    F.get_json_object(F.col("value"), "$.price").alias("price"),
    F.get_json_object(F.col("value"), "$.store").alias("brand"),
    F.get_json_object(F.col("value"), "$.parent_asin").alias("parent_asin")
)
df_meta.printSchema()

root
 |-- main_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- average_rating: string (nullable = true)
 |-- rating_number: string (nullable = true)
 |-- price: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- parent_asin: string (nullable = true)

In [16]:
df_meta.count()

213593

In [14]:
df_meta.show(10, truncate=False)

+-------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+---------------+-------+----------------------+-----------+
|m_main_category    |m_product_name                                                                                                                                                                                          |m_average_rating|m_rating_number|m_price|m_store               |parent_asin|
+-------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+---------------+-------+----------------------+-----------+
|Musical Instruments|Pearl Export Lacquer EXL725S/C249 5-Piece New Fusion Drum Set with Hardware, Honey

In [22]:
df_joined = df_review.join(
    df_meta,
    on="parent_asin",
    how="inner"
)

In [21]:
df_joined.show(5, truncate=False)

+-----------+----------+------------+------+------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [30]:
df = df_joined.select(
    "parent_asin",
    "asin",
    "helpful_vote",
    "rating",
    "text",
    "timestamp",
    "title",
    "user_id",
    "verified_purchase",
    "product_name",
    "price",
    "brand"
)

In [32]:
df.show(5, truncate=False)

+-----------+----------+------------+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [33]:
df.count()

3017439

In [37]:
df = df.fillna({
    "price": 0,
    "brand": "Not Available"
})

In [38]:
df = df.dropDuplicates()

In [39]:
print("Rows:", df.count())
print("Columns:", len(df.columns))
print("Unique Rows:", df.distinct().count())

Rows: 2983785
Columns: 12
Unique Rows: 2983785

In [40]:
from pyspark.sql.functions import from_unixtime, col

df = df.withColumn(
    "date",
    from_unixtime(col("timestamp") / 1000)
)

In [41]:
from pyspark.sql.functions import to_timestamp, year

df = df.withColumn("date", to_timestamp("date"))

df = df.withColumn("year", year("date"))

# Rating Distribution

In [42]:
df.groupBy("rating") \
  .count() \
  .orderBy("rating") \
  .show()

+------+-------+
|rating|  count|
+------+-------+
|   1.0| 260790|
|   2.0| 129900|
|   3.0| 194931|
|   4.0| 395928|
|   5.0|2002236|
+------+-------+

# Verified vs Non-verified

In [51]:
df.groupBy("verified_purchase") \
  .count() \
  .show()

+-----------------+-------+
|verified_purchase|  count|
+-----------------+-------+
|             true|2749402|
|            false| 234383|
+-----------------+-------+

In [50]:
df.groupBy("verified_purchase") \
  .avg("rating") \
  .show()

+-----------------+-----------------+
|verified_purchase|      avg(rating)|
+-----------------+-----------------+
|             true|4.270390797708011|
|            false|4.092677369945773|
+-----------------+-----------------+

# Review Length Analysis

In [49]:
from pyspark.sql.functions import length
from pyspark.sql.functions import col

df = df.withColumn(
    "review_length",
    length(col("text"))
)

df.groupBy("rating") \
  .avg("review_length") \
  .show()

+------+------------------+
|rating|avg(review_length)|
+------+------------------+
|   1.0| 251.7893017370298|
|   4.0| 320.9427168576105|
|   3.0|326.09881445229337|
|   2.0|327.33079291762897|
|   5.0|206.40616940260787|
+------+------------------+

# Time-Based Analysis

In [52]:
df.groupBy("year") \
  .count() \
  .orderBy("year") \
  .show(30)

+----+------+
|year| count|
+----+------+
|1999|     2|
|2000|    29|
|2001|    45|
|2002|    69|
|2003|   186|
|2004|   424|
|2005|   771|
|2006|  1197|
|2007|  2988|
|2008|  4318|
|2009|  7545|
|2010| 16808|
|2011| 32042|
|2012| 51193|
|2013|115528|
|2014|172282|
|2015|239309|
|2016|268160|
|2017|258602|
|2018|270652|
|2019|342719|
|2020|386070|
|2021|385836|
|2022|297684|
|2023|129326|
+----+------+

# Sentiment Count

In [53]:
from pyspark.sql.functions import when

df = df.withColumn(
    "sentiment",
    when(df.rating >= 4, "Positive")
    .when(df.rating == 3, "Neutral")
    .otherwise("Negative")
)

df.groupBy("sentiment").count().show()

+---------+-------+
|sentiment|  count|
+---------+-------+
| Positive|2398164|
|  Neutral| 194931|
| Negative| 390690|
+---------+-------+

# Helpful Votes Analysis

In [54]:
df.orderBy(df.helpful_vote.desc()) \
  .select("title","rating","helpful_vote") \
  .show(20, truncate=False)

+--------------------------------------------------------------------------------------+------+------------+
|title                                                                                 |rating|helpful_vote|
+--------------------------------------------------------------------------------------+------+------------+
|Great quality microphone                                                              |5.0   |4650        |
|Great mic for the price!                                                              |5.0   |4234        |
|An incredible microphone packed full of features - for an amazing price [VIDEO]       |5.0   |3158        |
|Great Starter Package                                                                 |4.0   |2647        |
|Hours of driving my husband crazy!                                                    |5.0   |2541        |
|READ Before you write another bad review!                                             |5.0   |2370        |
|Only $53???!!!  A 

# Product Popularity

In [56]:
df.groupBy("parent_asin","product_name") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(20, truncate=False)

+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|parent_asin|product_name                                                                                                                                                                                       |count|
+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|B09857JRP2 |GLS Audio Instrument Cable - Amp Cord for Bass & Electric Guitar - Straight to Right Angle 1/4 Inch Instrument Cable - Black/Grey Braided Tweed, 10ft                                              |9245 |
|B09W4F2X6S |BONAOK Wireless Bluetooth Karaoke Microphone, 3-in-1 Portable Handheld Mic Speaker Machine for All Smartphones, Gifts to Gi

# Top Brand Analysis

In [57]:
df.groupBy("brand") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+--------------------+-----+
|               brand|count|
+--------------------+-----+
|              Fender|72677|
|                Pyle|59943|
|              Donner|50763|
|              YAMAHA|44550|
|           D'Addario|40519|
|          JIM DUNLOP|38602|
|           Behringer|32901|
|             OnStage|32262|
|               SNARK|27905|
|D'Addario Accesso...|26004|
|          Ernie Ball|24688|
|              Neewer|22845|
|          ChromaCast|20580|
|           GLS Audio|20304|
|      Audio-Technica|19823|
|               Shure|18422|
|         Hola! Music|17556|
|  Mendini by Cecilio|14527|
|               Gator|14010|
|Best Choice Products|13882|
+--------------------+-----+
only showing top 20 rows